<a href="https://colab.research.google.com/github/kousiknandy/pycolab/blob/main/serialize_kv_store.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [114]:
import random
import string

def generate_random_full_string(length, weights):
    characters = weights[0] * string.ascii_letters + weights[1] * string.digits \
    + weights[2] * string.punctuation
    return ''.join(random.choice(characters) for i in range(length))

random_dict = {}
for _ in range(10):
    key = generate_random_full_string(6, (2, 1, 0))
    value = generate_random_full_string(10, (4, 2, 1))
    random_dict[key] = value

In [115]:
random_dict

{'HXjtId': 'kkG}X)M;Jj',
 'QsDkGT': '{GgrYgXmiN',
 'LVaE3u': 'HYP~ct5aof',
 '8IjjNi': 'OjUN(Cr*g@',
 'MmpIMG': 'Ptil^oG(=x',
 'AxnMp4': 'QokNS8IZ5x',
 'XnMurM': 'clVY.)/WBW',
 'AlsjYm': 'S8daNDE5vp',
 'oyXOxy': 'hors*VLn{Y',
 'CdNQm1': "-ZhpJX'f;i"}

In [116]:
from struct import pack_into, unpack_from

class FieldEncoder:
    def __init__(self, small: bool):
        self.small = small

    def encode_field(self, header: int, data: str):
        h = bytearray(5)
        b = data.encode("utf-8")
        pack_into("<BL", h, 0, header, len(b))
        return h + b

    def decode_field(self, blob):
        data = memoryview(blob)
        h, l = unpack_from("<BL", data)
        return  l+5, h, bytearray(data[5:l+5]).decode("utf-8")

In [117]:
from itertools import islice

class SerDe:
    def __init__(self, kv_store):
        self.fe = FieldEncoder(False)
        self.kv_store = kv_store

    def serialize(self):
        for k, v in self.kv_store.items():
            yield self.fe.encode_field(1, k)
            yield self.fe.encode_field(2, v)

    def deserialize(self, blob):
        blob = memoryview(blob)
        off = 0
        while off < len(blob):
            l, _, key = self.fe.decode_field(blob[off:])
            off += l
            l, _, val = self.fe.decode_field(blob[off:])
            off += l
            yield key, val

    def deserialize_stream(self, stream):
        while True:
            hdr = bytearray(islice(stream, 5))
            if len(hdr) == 0: return
            h, l = unpack_from("<BL", hdr)
            key = bytearray(islice(stream, l)).decode("utf-8")
            hdr = bytearray(islice(stream, 5))
            h, l = unpack_from("<BL", hdr)
            val = bytearray(islice(stream, l)).decode("utf-8")
            yield key, val

In [118]:
sd = SerDe(random_dict)
sink = bytearray()
for by in sd.serialize():
    sink += by
print(sink)
output_dict = {k: v for k, v in sd.deserialize(sink)}
print(random_dict)
print(output_dict)

bytearray(b"\x01\x06\x00\x00\x00HXjtId\x02\n\x00\x00\x00kkG}X)M;Jj\x01\x06\x00\x00\x00QsDkGT\x02\n\x00\x00\x00{GgrYgXmiN\x01\x06\x00\x00\x00LVaE3u\x02\n\x00\x00\x00HYP~ct5aof\x01\x06\x00\x00\x008IjjNi\x02\n\x00\x00\x00OjUN(Cr*g@\x01\x06\x00\x00\x00MmpIMG\x02\n\x00\x00\x00Ptil^oG(=x\x01\x06\x00\x00\x00AxnMp4\x02\n\x00\x00\x00QokNS8IZ5x\x01\x06\x00\x00\x00XnMurM\x02\n\x00\x00\x00clVY.)/WBW\x01\x06\x00\x00\x00AlsjYm\x02\n\x00\x00\x00S8daNDE5vp\x01\x06\x00\x00\x00oyXOxy\x02\n\x00\x00\x00hors*VLn{Y\x01\x06\x00\x00\x00CdNQm1\x02\n\x00\x00\x00-ZhpJX\'f;i")
{'HXjtId': 'kkG}X)M;Jj', 'QsDkGT': '{GgrYgXmiN', 'LVaE3u': 'HYP~ct5aof', '8IjjNi': 'OjUN(Cr*g@', 'MmpIMG': 'Ptil^oG(=x', 'AxnMp4': 'QokNS8IZ5x', 'XnMurM': 'clVY.)/WBW', 'AlsjYm': 'S8daNDE5vp', 'oyXOxy': 'hors*VLn{Y', 'CdNQm1': "-ZhpJX'f;i"}
{'HXjtId': 'kkG}X)M;Jj', 'QsDkGT': '{GgrYgXmiN', 'LVaE3u': 'HYP~ct5aof', '8IjjNi': 'OjUN(Cr*g@', 'MmpIMG': 'Ptil^oG(=x', 'AxnMp4': 'QokNS8IZ5x', 'XnMurM': 'clVY.)/WBW', 'AlsjYm': 'S8daNDE5vp', 'oyXOxy': 

In [119]:
from itertools import chain, islice

class Chunker:
    def __init__(self, size):
        self.size = size
        self.count = 0

    def cut(self, pieces):
        stream = chain.from_iterable(pieces)
        while True:
            chunk = bytearray(self.size)
            data = list(islice(stream, self.size - 8))
            if len(data) == 0: return
            pack_into(">LL", chunk, 0, self.count, len(data))
            chunk[8:] = data
            self.count += 1
            yield chunk

    def join(self, pieces):
        yield from chain.from_iterable(p[8:] for p in pieces)

In [120]:
sd = SerDe(random_dict)
ch = Chunker(48)
p = sd.serialize()
p = ch.cut(p)
p = ch.join(p)
output_dict = {k: v for k, v in sd.deserialize_stream(p)}
print(output_dict)

{'HXjtId': 'kkG}X)M;Jj', 'QsDkGT': '{GgrYgXmiN', 'LVaE3u': 'HYP~ct5aof', '8IjjNi': 'OjUN(Cr*g@', 'MmpIMG': 'Ptil^oG(=x', 'AxnMp4': 'QokNS8IZ5x', 'XnMurM': 'clVY.)/WBW', 'AlsjYm': 'S8daNDE5vp', 'oyXOxy': 'hors*VLn{Y', 'CdNQm1': "-ZhpJX'f;i"}
